
# Step 0 — Install

In [2]:
# ============================================================
!pip install git+https://github.com/facebookresearch/segment-anything.git -q
!pip install opencv-python-headless matplotlib numpy Pillow -q

import os
if not os.path.exists('sam_vit_h.pth'):
    !wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth -O sam_vit_h.pth
    print('SAM checkpoint downloaded ✅')
else:
    print('SAM checkpoint already exists ✅')

  Preparing metadata (setup.py) ... done
SAM checkpoint downloaded ✅



# Step 1 — Mount Drive & Config

In [3]:
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

CONFIG = {
    "image_dir":      "/content/drive/MyDrive/bacterial_leaf_blight",
    "output_dir":     "/content/drive/MyDrive/blb_sam_output",
    "mask_dir":       "/content/drive/MyDrive/blb_sam_masks",
    "sam_checkpoint": "/content/sam_vit_h.pth",
    "sam_model_type": "vit_h",
    "display_size":   640,
}

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png")
os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(CONFIG["mask_dir"],   exist_ok=True)
print("Config ready ✅")

Mounted at /content/drive
Config ready ✅


# Step 2 — Load SAM

In [4]:
# ============================================================
import torch, numpy as np, cv2, os, json, time
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from PIL import Image
from io import BytesIO
import base64
from segment_anything import sam_model_registry, SamPredictor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

sam = sam_model_registry[CONFIG["sam_model_type"]](checkpoint=CONFIG["sam_checkpoint"])
sam.to(device)
predictor = SamPredictor(sam)
print("SAM loaded ✅")

Device: cpu
SAM loaded ✅


# Step 3 — Annotate

In [10]:
# ============================================================

# ── Helpers ────────────────────────────────────────────────

def get_display_size(img_rgb, max_size=640):
    h, w  = img_rgb.shape[:2]
    scale = min(max_size / w, max_size / h, 1.0)
    return int(w * scale), int(h * scale), scale

def img_to_b64(img_rgb, size=None):
    pil = Image.fromarray(img_rgb)
    if size:
        pil = pil.resize(size, Image.LANCZOS)
    buf = BytesIO()
    pil.save(buf, format='JPEG', quality=90)
    return base64.b64encode(buf.getvalue()).decode()

def run_sam_multibox(img_rgb, boxes, scale):
    predictor.set_image(img_rgb)
    combined = np.zeros(img_rgb.shape[:2], dtype=np.uint8)
    for box in boxes:
        x0 = int(min(box[0], box[2]) / scale)
        y0 = int(min(box[1], box[3]) / scale)
        x1 = int(max(box[0], box[2]) / scale)
        y1 = int(max(box[1], box[3]) / scale)
        masks, scores, _ = predictor.predict(
            box=np.array([x0, y0, x1, y1]),
            multimask_output=True,
        )
        best = masks[np.argmax(scores)].astype(np.uint8) * 255
        combined = cv2.bitwise_or(combined, best)
    return combined

def save_outputs(img_rgb, mask, stem):
    overlay = img_rgb.copy()
    overlay[mask > 127] = [255, 0, 0]
    cv2.imwrite(os.path.join(CONFIG["mask_dir"], stem + "_mask.png"), mask)
    side = np.hstack([img_rgb, cv2.cvtColor(mask, cv2.COLOR_GRAY2RGB), overlay])
    cv2.imwrite(
        os.path.join(CONFIG["output_dir"], stem + "_sidebyside.jpg"),
        cv2.cvtColor(side, cv2.COLOR_RGB2BGR))


# ── File list ─────────────────────────────────────────────

all_files = sorted([
    f for f in os.listdir(CONFIG["image_dir"])
    if f.lower().endswith(IMG_EXTENSIONS)
])
done_stems = set(
    f.replace('_mask.png', '')
    for f in os.listdir(CONFIG["mask_dir"]) if f.endswith('_mask.png')
)
pending = [f for f in all_files if os.path.splitext(f)[0] not in done_stems]
print(f"Total: {len(all_files)}  |  Done: {len(done_stems)}  |  Pending: {len(pending)}")


# ── State ─────────────────────────────────────────────────
state = {
    'idx': 0, 'img_rgb': None, 'mask': None,
    'scale': 1.0, 'stem': '', 'dw': 640, 'dh': 480,
    'b64': '', 'canvas_id': 0   # <-- unique ID incremented on every load
}


# ── Widgets ───────────────────────────────────────────────
progress_lbl = widgets.HTML()
status_lbl   = widgets.HTML()
boxes_lbl    = widgets.HTML(value='<b>Boxes:</b> none')

box_store = widgets.Textarea(value='[]', layout=widgets.Layout(display='none'))

canvas_out = widgets.Output()
result_out = widgets.Output()

btn_sam    = widgets.Button(description='▶ Run SAM',         button_style='primary', disabled=True)
btn_save   = widgets.Button(description='✅ Save',            button_style='success', disabled=True)
btn_skip   = widgets.Button(description='⏭ Skip',            button_style='warning')
btn_redo   = widgets.Button(description='↩ Redo All Boxes',  button_style='',        disabled=True)
btn_delete = widgets.Button(description='❌ Delete Selected', button_style='danger',  disabled=True)


# ── Canvas renderer ───────────────────────────────────────
# FIX 1: Each canvas gets a unique `canvas_id` so stale scripts from
#         previous images bail out immediately on the `getElementById` check.
# FIX 2: Use img.onload instead of a polling retry loop — reliable and instant.
# FIX 3: pushBoxes targets only the textarea that belongs to THIS canvas's
#         output cell (matched by the unique marker span), not all textareas.

def make_canvas_html(b64, dw, dh, canvas_id):
    return f"""
<style>
  #ann-canvas-{canvas_id} {{ cursor:crosshair; border:2px solid #555; display:block; background:#222; }}
  #ann-info-{canvas_id}   {{ font-family:monospace; font-size:12px; margin-top:5px; color:#aaa; }}
</style>
<canvas id="ann-canvas-{canvas_id}" width="{dw}" height="{dh}"></canvas>
<div id="ann-info-{canvas_id}">Loading image...</div>
<script>
(function(){{
  // Guard: if our canvas is not in the DOM this is a stale script — bail immediately.
  const canvas = document.getElementById('ann-canvas-{canvas_id}');
  if (!canvas) return;

  const ctx  = canvas.getContext('2d');
  const info = document.getElementById('ann-info-{canvas_id}');

  let boxes    = [];
  let selected = -1;
  let drawing  = false;
  let startX, startY, curX, curY;

  // ── Image loading via onload (reliable, no polling) ──
  const img = new Image();
  img.onload = function() {{
    ctx.drawImage(img, 0, 0);
    info.textContent = 'Click and drag to draw a box. Click a box to select it (yellow).';
  }};
  img.onerror = function() {{
    info.textContent = 'ERROR: image failed to load.';
  }};
  img.src = 'data:image/jpeg;base64,{b64}';

  function redraw() {{
    ctx.clearRect(0, 0, canvas.width, canvas.height);
    if (img.complete && img.naturalWidth > 0) ctx.drawImage(img, 0, 0);
    boxes.forEach((b, i) => {{
      const sel = (i === selected);
      ctx.strokeStyle = sel ? '#FFD700' : '#FF3333';
      ctx.lineWidth   = sel ? 3 : 2;
      ctx.strokeRect(b.x0, b.y0, b.x1 - b.x0, b.y1 - b.y0);
      ctx.fillStyle = sel ? '#FFD700' : '#FF3333';
      ctx.font = 'bold 13px monospace';
      ctx.fillText('#' + (i+1), b.x0 + 4, b.y0 + 15);
    }});
    if (drawing) {{
      ctx.strokeStyle = '#00EEFF';
      ctx.lineWidth   = 2;
      ctx.setLineDash([5, 3]);
      ctx.strokeRect(startX, startY, curX - startX, curY - startY);
      ctx.setLineDash([]);
    }}
  }}

  // ── Push boxes to Python via textarea ───────────────────
  // FIX: walk UP from our canvas to find the nearest textarea sibling,
  // rather than blindly querying all textareas on the page.
  function pushBoxes() {{
    // Bail if our canvas was removed (image navigated away)
    if (!document.getElementById('ann-canvas-{canvas_id}')) return;

    const arr  = boxes.map(b => [b.x0, b.y0, b.x1, b.y1]);
    const json = JSON.stringify(arr);

    // Walk up to the output cell container, then find textarea within it
    let el = canvas;
    let ta = null;
    for (let i = 0; i < 15; i++) {{
      el = el.parentElement;
      if (!el) break;
      ta = el.querySelector('textarea');
      if (ta) break;
    }}
    if (!ta) {{
      // Fallback: target the LAST textarea (most recently rendered widget)
      const all = document.querySelectorAll('textarea');
      if (all.length) ta = all[all.length - 1];
    }}
    if (ta) {{
      ta.value = json;
      ['input','change'].forEach(ev =>
        ta.dispatchEvent(new Event(ev, {{bubbles: true}})));
    }}

    info.textContent = boxes.length + ' box(es). Selected: ' +
      (selected >= 0 ? '#'+(selected+1) : 'none') +
      ' | Click box to select, drag empty area to draw new box.';
  }}

  function getPos(e) {{
    const r = canvas.getBoundingClientRect();
    return [e.clientX - r.left, e.clientY - r.top];
  }}

  function hitBox(x, y) {{
    for (let i = boxes.length - 1; i >= 0; i--) {{
      const b  = boxes[i];
      const x0 = Math.min(b.x0, b.x1), x1 = Math.max(b.x0, b.x1);
      const y0 = Math.min(b.y0, b.y1), y1 = Math.max(b.y0, b.y1);
      if (x >= x0 && x <= x1 && y >= y0 && y <= y1) return i;
    }}
    return -1;
  }}

  canvas.addEventListener('mousedown', e => {{
    const [x, y] = getPos(e);
    const hit = hitBox(x, y);
    if (hit >= 0) {{
      selected = (selected === hit) ? -1 : hit;
      redraw(); pushBoxes();
    }} else {{
      selected = -1;
      startX = x; startY = y; drawing = true;
    }}
  }});

  canvas.addEventListener('mousemove', e => {{
    if (!drawing) return;
    [curX, curY] = getPos(e);
    redraw();
  }});

  canvas.addEventListener('mouseup', e => {{
    if (!drawing) return;
    drawing = false;
    const [x, y] = getPos(e);
    if (Math.abs(x - startX) > 8 && Math.abs(y - startY) > 8)
      boxes.push({{x0: startX, y0: startY, x1: x, y1: y}});
    redraw(); pushBoxes();
  }});

  // Expose delete/clear helpers scoped to this canvas_id to avoid collisions
  window['_ann_delete_{canvas_id}'] = function() {{
    if (selected >= 0) {{ boxes.splice(selected, 1); selected = -1; }}
    redraw(); pushBoxes();
  }};
  window['_ann_clear_{canvas_id}'] = function() {{
    boxes = []; selected = -1; redraw(); pushBoxes();
  }};

  // Keep global aliases pointing to the CURRENT canvas always
  window._ann_delete_selected = window['_ann_delete_{canvas_id}'];
  window._ann_clear_all       = window['_ann_clear_{canvas_id}'];

}})();
</script>
"""


# ── Load image ────────────────────────────────────────────

def load_image(idx):
    if idx >= len(pending):
        with canvas_out:
            clear_output()
            print("🎉 All images annotated!")
        progress_lbl.value = "<b>Done!</b>"
        for b in [btn_sam, btn_save, btn_skip, btn_redo, btn_delete]:
            b.disabled = True
        return

    fname   = pending[idx]
    img_bgr = cv2.imread(os.path.join(CONFIG["image_dir"], fname))
    if img_bgr is None:
        status_lbl.value = f"⚠️ Could not read {fname}, skipping."
        state['idx'] += 1
        load_image(state['idx'])
        return

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    dw, dh, scale = get_display_size(img_rgb, CONFIG["display_size"])
    b64 = img_to_b64(img_rgb, size=(dw, dh))

    # Increment canvas_id so the new canvas HTML has a unique ID
    # and any JS from the previous image will bail on its getElementById guard.
    state['canvas_id'] += 1

    state.update({
        'img_rgb': img_rgb, 'scale': scale,
        'stem': os.path.splitext(fname)[0],
        'mask': None, 'dw': dw, 'dh': dh, 'b64': b64
    })

    progress_lbl.value = (
        f"<b>Image {idx+1} / {len(pending)}</b> &nbsp;|&nbsp; <code>{fname}</code>"
    )
    status_lbl.value  = "Draw boxes around the lesion(s)."
    boxes_lbl.value   = "<b>Boxes:</b> none"
    box_store.value   = '[]'

    btn_sam.disabled    = True
    btn_save.disabled   = True
    btn_redo.disabled   = True
    btn_delete.disabled = True

    with result_out:
        clear_output()
    with canvas_out:
        clear_output(wait=True)
        display(HTML(make_canvas_html(b64, dw, dh, state['canvas_id'])))


# ── Box store observer ────────────────────────────────────

def on_box_change(change):
    try:
        boxes = json.loads(change['new'])
    except:
        return
    n = len(boxes)
    boxes_lbl.value     = f"<b>Boxes drawn:</b> {n}" if n else "<b>Boxes:</b> none"
    btn_sam.disabled    = (n == 0)
    btn_redo.disabled   = (n == 0)
    btn_delete.disabled = (n == 0)
    btn_save.disabled   = True
    state['mask']       = None

box_store.observe(on_box_change, names='value')


# ── Button handlers ───────────────────────────────────────

def on_run_sam(b):
    try:
        boxes = json.loads(box_store.value)
    except:
        boxes = []
    if not boxes:
        status_lbl.value = "⚠️ Draw at least one box first!"
        return

    status_lbl.value = f"⏳ Running SAM on {len(boxes)} box(es)..."
    btn_sam.disabled = True

    mask = run_sam_multibox(state['img_rgb'], boxes, state['scale'])
    state['mask'] = mask

    img_rgb = state['img_rgb']
    overlay = img_rgb.copy()
    overlay[mask > 127] = [255, 0, 0]

    with result_out:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        axes[0].imshow(img_rgb);           axes[0].set_title('Original');  axes[0].axis('off')
        axes[1].imshow(mask, cmap='gray'); axes[1].set_title('SAM Mask'); axes[1].axis('off')
        axes[2].imshow(overlay);           axes[2].set_title('Overlay');  axes[2].axis('off')
        plt.tight_layout()
        plt.show()

    status_lbl.value  = "✔ SAM done — Save if correct, or add/delete boxes and run again."
    btn_save.disabled = False
    btn_sam.disabled  = False


def on_save(b):
    if state['mask'] is None:
        status_lbl.value = "⚠️ Run SAM first!"
        return
    save_outputs(state['img_rgb'], state['mask'], state['stem'])
    status_lbl.value = f"✅ Saved {state['stem']}"
    state['idx'] += 1
    load_image(state['idx'])


def on_skip(b):
    state['idx'] += 1
    load_image(state['idx'])


def on_redo(b):
    state['mask']     = None
    btn_save.disabled = True
    with result_out:
        clear_output()
    with canvas_out:
        clear_output(wait=True)
        display(HTML(make_canvas_html(state['b64'], state['dw'], state['dh'], state['canvas_id'])))
    status_lbl.value = "Canvas reset — draw new boxes."


def on_delete(b):
    cid = state['canvas_id']
    display(HTML(
        f"<script>if(window['_ann_delete_{cid}']) window['_ann_delete_{cid}']();</script>"
    ))


btn_sam.on_click(on_run_sam)
btn_save.on_click(on_save)
btn_skip.on_click(on_skip)
btn_redo.on_click(on_redo)
btn_delete.on_click(on_delete)


# ── Render UI ─────────────────────────────────────────────

ui = widgets.VBox([
    progress_lbl,
    status_lbl,
    boxes_lbl,
    box_store,
    canvas_out,
    widgets.HBox([btn_sam, btn_save, btn_skip, btn_redo, btn_delete]),
    result_out,
])
display(ui)
state['idx'] = 0
load_image(0)

Total: 1094  |  Done: 109  |  Pending: 985


# Progress Check  (run this cell separately anytime)

In [9]:
# ============================================================
total = len([f for f in os.listdir(CONFIG['image_dir']) if f.lower().endswith(IMG_EXTENSIONS)])
done  = len([f for f in os.listdir(CONFIG['mask_dir'])  if f.endswith('_mask.png')])
print(f"Total   : {total}")
print(f"Done    : {done}")
print(f"Left    : {total - done}")
print(f"Progress: {done/total*100:.1f}%")

Total   : 1094
Done    : 109
Left    : 985
Progress: 10.0%
